In [ ]:
import os
import json
import re

class ExactCorpusMatchPipeline:
    def __init__(self, lexicon_path="Resources/Lexicon.txt"):
        self.lexicon_path = lexicon_path
        self.dictionary = {}
        
        if os.path.exists(self.lexicon_path):
            self.build_model_ready_dictionary()
        else:
            raise FileNotFoundError(f"Could not find the lexicon file at: {os.path.abspath(self.lexicon_path)}. Make sure it is inside the Resources folder.")

    def clean_and_align_to_corpus(self, raw_translit: str) -> str:
        """
        Converts Lexicon entries to exactly match the target corpus layout:
        Example: "=k" -> " .k" 
        """
        cleaned = raw_translit
        
        # 1. Base character adjustments from her notebook rules
        cleaned = cleaned.replace('z', 's')
        
        # 2. Map Suffix markers: fayrose corpus uses " .f" or " .k" instead of "=f" or "=k"
        cleaned = cleaned.replace('=', ' .')
        
        # 3. Handle standalone punctuation dots or hyphens:
        # Ensure dots have spaces around them so OpenNMT treats them as separate tokens
        cleaned = cleaned.replace('.', ' . ')
        cleaned = cleaned.replace('-', ' ') 
        cleaned = cleaned.replace('*', '')  
        
        # 4. Clean up lines and fix spacing duplicates
        cleaned = cleaned.replace('\r\n', ' ').replace('\n', ' ')
        cleaned = re.sub(r'\s{2,}', ' ', cleaned)
        
        return cleaned.strip()

    def build_model_ready_dictionary(self):
        """Parses the real Lexicon.txt to form an in-memory dictionary cache matching corpus styles"""
        print(f"Compiling your real raw database from {self.lexicon_path}...")
        
        with open(self.lexicon_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                
                parts = line.split(';')
                if len(parts) >= 3:
                    raw_gardiner = parts[0]
                    raw_translit = parts[1]
                    raw_translation = parts[2]
                    
                    # Process Gardiner key token list: "A12,A1,Z2," -> "A12 A1 Z2"
                    tokens = [t.strip().upper() for t in raw_gardiner.split(',') if t.strip()]
                    gardiner_key = " ".join(tokens)
                    
                    # Convert to exact OpenNMT training text format
                    model_ready_translit = self.clean_and_align_to_corpus(raw_translit)
                    
                    if gardiner_key and model_ready_translit:
                        if gardiner_key not in self.dictionary:
                            self.dictionary[gardiner_key] = {
                                "model_input": model_ready_translit,
                                "translation_hint": raw_translation.strip()
                            }
                            
        print(f"Successfully loaded {len(self.dictionary)} vocabulary keys into your pipeline!")

    def clean_and_align_to_corpus(self, raw_translit: str) -> str:
        """
        Strips all punctuation (. = - *) completely and matches 
        the clean, space-separated token layout.
        """
        cleaned = raw_translit
        
        # 1. Base character adjustments (z -> s)
        cleaned = cleaned.replace('z', 's')
        
        # 2. Obliterate ALL punctuation characters completely
        cleaned = cleaned.replace('=', ' ').replace('.', ' ').replace('-', ' ').replace('*', '')
        
        # 3. Clean up linebreaks and squash multiple spaces
        cleaned = cleaned.replace('\r\n', ' ').replace('\n', ' ')
        cleaned = re.sub(r'\s{2,}', ' ', cleaned)
        
        return cleaned.strip()

    def process_gardiner_list(self, extracted_list: list) -> dict:
        """Processes input lists of Gardiner codes into dot-free OpenNMT strings"""
        cleaned_input_tokens = [str(item).strip().upper() for item in extracted_list if str(item).strip()]
        full_query_key = " ".join(cleaned_input_tokens)
        
        # 1. Check for exact phrase sequence match
        if full_query_key in self.dictionary:
            entry = self.dictionary[full_query_key]
            return {
                "input_sequence": full_query_key,
                "strategy": "exact_phrase_match",
                "open_nmt_ready_string": entry["model_input"],
                "english_hint": entry["translation_hint"]
            }
            
        # 2. Pure Space-Separated Sliding Window Fallback
        resolved_tokens = []
        resolved_hints = []
        i = 0
        while i < len(cleaned_input_tokens):
            matched = False
            for width in range(4, 0, -1):
                if i + width <= len(cleaned_input_tokens):
                    chunk = " ".join(cleaned_input_tokens[i:i+width])
                    if chunk in self.dictionary:
                        resolved_tokens.append(self.dictionary[chunk]["model_input"])
                        resolved_hints.append(self.dictionary[chunk]["translation_hint"])
                        i += width
                        matched = True
                        break
            if not matched:
                resolved_tokens.append("UNK")
                i += 1
                
        final_string = " ".join(resolved_tokens)
        final_string = re.sub(r'\s{2,}', ' ', final_string)
        
        return {
            "input_sequence": full_query_key,
            "strategy": "sliding_window_fallback",
            "open_nmt_ready_string": final_string.strip(),
            "english_hint": " / ".join(resolved_hints) if resolved_hints else "Unknown Context"
        }

# ==================== RUN TESTS ON REAL DATA ====================
if __name__ == "__main__":
    try:
        pipeline = ExactCorpusMatchPipeline("Resources/Lexicon.txt")
        print("-" * 60)
        
        test_cases = {
            "1. Standard Word Match": ["A12", "A1", "Z2"],
            "2. Suffix Pronoun Match": ["A1"],
            "3. Verbal Sentence ('The scribe hears')": ["F21", "M17", "M17", "Y1", "Y1", "A1", "Z2"],
            "4. Noun + Suffix Pronoun ('His army')": ["A12", "A1", "Z2", "I9"],
            "5. Prepositional Phrase ('In the house')": ["G17", "O1", "Z1"],
            "6. Robustness Test (Unknown Sign handling)": ["A12", "A1", "Z2", "INVALID_SIGN", "I9"],
            "4. Adjectival Sentence ('The god is great')": ["O29", "D21", "N35", "R8", "Z1"],
            "test sentence": ['J11', 'D58', 'D26', 'X1']
        }
        
        for name, sign_list in test_cases.items():
            print(f"Running: {name}")
            result = pipeline.process_gardiner_list(sign_list)
            print(f"OpenNMT Input: {result['open_nmt_ready_string']}")
            print(f"English Hints: {result['english_hint']}")
            print("-" * 60)

    except Exception as e:
        print(f"An error occurred: {e}")

Compiling your real raw database from Resources/Lexicon.txt...
Successfully loaded 10753 vocabulary keys into your pipeline!
------------------------------------------------------------
Running: 1. Standard Word Match
OpenNMT Input: mSa
English Hints: soldiers, army, infantry, gang (of workmen)
------------------------------------------------------------
Running: 2. Suffix Pronoun Match
OpenNMT Input: i
English Hints: I, me, my
------------------------------------------------------------
Running: 3. Verbal Sentence ('The scribe hears')
OpenNMT Input: sDm sXA
English Hints: the scribe hears
------------------------------------------------------------
Running: 4. Noun + Suffix Pronoun ('His army')
OpenNMT Input: mSa f
English Hints: soldiers, army, infantry, gang (of workmen) / he, him, his, it, its
------------------------------------------------------------
Running: 5. Prepositional Phrase ('In the house')
OpenNMT Input: m pr
English Hints: with, by means of, from, out of, as, namely, 